# Inference FLOPs: H-balanced model vs SAM

Companion to [`../concepts/h-balanced-model.md`](../concepts/h-balanced-model.md) and
[`../concepts/sam-model.md`](../concepts/sam-model.md).

An H-balanced transformer is fixed by one even integer `H`:

| | rule |
|---|---|
| layers | `n_blocks = H` |
| heads | `q_heads = H`, `kv_heads = H/2` (GQA, always) |
| residual width | `dmodel = 64 H`, so `head_dim = 64` |
| FF hidden | `dff = 4 dmodel = 256 H`, gated (`gate`, `up`, `down`) |
| vocab | GPT-NeoX tokenizer, ~50k |

**SAM** (stateless memory attention, in development) is the same model **without the `q_proj` and
`k_proj` weights** and, with no q and k vectors, **without any attention score cost**: its FLOPs
are purely linear in tokens. Pass `variant="sam"` to count it. If q/k vectors turn out to be
produced some other way, `attention_cost=c` charges `c x` the vanilla score path (`c = 0` by
default). How its attention works is otherwise out of scope here.

**SAMm** (`variant="samm"`) is SAM with the freed `q_proj`/`k_proj` weights spent on a wider FF
instead of given back - `dff = 288 H = 4.5 dmodel`, exactly the vanilla parameter count. It appears
in the last two figures only.

## The counting model

**Weights: 2 FLOPs per active parameter per token** (one multiply + one add per MAC).
Active = every parameter that takes part in a matmul for that token:

* attention projections (`q, k, v, o` for the vanilla model, `v, o` for SAM) and the three gated-FF
  matrices (`gate`, `up`, `down`), in every layer;
* the LM head (`vocab x dmodel`);
* **not** the embedding table - a lookup is a gather, not a matmul, so it contributes
  parameters but no FLOPs.

**Attention is counted exactly, not as "2 x params".** The score path has no weights; its cost is
quadratic in the context and is what makes long contexts expensive. With causal masking only the
`i >= j` pairs are computed - for the vanilla model, and for SAM only in proportion to
`attention_cost`:

```
pairs(L)  = L(L+1)/2                       # prefill over a context of L tokens
pairs     = L                              # one decode step attending to L cached keys

QK^T  : 2 * head_dim FLOPs per (query, key, head)   -> 2 * head_dim * q_heads * pairs
A @ V : 2 * head_dim FLOPs per (query, key, head)   -> 2 * head_dim * q_heads * pairs
            per layer: 4 * datt * pairs            (datt = head_dim * q_heads)
```

GQA does **not** reduce the score path: the `H/2` kv heads are broadcast to all `H` query heads, so
it runs `q_heads` times. GQA only shrinks the `k`/`v` projections (and the KV cache).

**Excluded** (together below ~1% at any `H` we use, and all linear in tokens): RMSNorm/LayerNorm,
residual additions, the RoPE rotation, the activation function, and the softmax itself -
the last of these is available behind `softmax_flops_per_score` if you want it counted.

In [ ]:
from dataclasses import dataclass, asdict
from math import ceil, log10

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

HEAD_DIM = 64  # fixed by the H-balanced rule: dmodel = 64H and q_heads = H
VARIANTS = ("vanilla", "sam", "samm")

## Configuration

Everything you would want to change lives in the next cell. Edit it, run the notebook top to
bottom, and every figure, table and check below follows - `SAM_ATTENTION_COST` in particular, which
decides how much of the vanilla attention score path SAM is charged for.

In [ ]:
# ----- the one cell to edit -------------------------------------------------------------------
SAM_ATTENTION_COST = 0.0     # fraction of the vanilla score path charged to SAM: 0 = none, 1 = full
H_VALUES = [16, 20, 24, 32, 40, 50]          # model sizes drawn in every figure
CONTEXT_RANGE = (256, 1 << 20)               # span of the context axis: 256 .. 1,048,576 tokens
CONTEXT_POINTS = 100         # log-spaced samples across it, so hovering reads values in between
VOCAB = 50_257               # GPT-NeoX tokenizer, as in configs/_model/llmrandom/*
GATED_FF = True              # gate + up + down; False counts the ungated two-matrix FF
H_PICK = 16                  # the size broken down in the "where the FLOPs go" figure
CONTEXTS_BD = [1024, 8192, 65536, 1048576]   # the contexts that figure shows
SWEEP_COSTS = [0.01, 0.02, 0.04, 0.08, 0.16, 0.32]   # attention-cost lines in the sweep figures
SWEEP_PAIRS = [(50, 50), (50, 40), (100, 80), (252, 200)]  # (SAM H, vanilla H) drawn in the sweeps
METHOD_PAIR = (100, 80)      # (SAM H, vanilla H) for the long-context attention-method figure
THEME = "light"              # "light" or "dark"
# ------------------------------------------------------------------------------------------------

_ratio = CONTEXT_RANGE[1] / CONTEXT_RANGE[0]
CONTEXTS = sorted({int(round(CONTEXT_RANGE[0] * _ratio ** (i / (CONTEXT_POINTS - 1))))
                   for i in range(CONTEXT_POINTS)})          # the sampled x values
CONTEXT_TICKS = [2 ** k for k in range(4, 31)                 # ... labelled at the powers of two
                 if CONTEXT_RANGE[0] <= 2 ** k <= CONTEXT_RANGE[1]]

VANILLA = "vanilla"
SAM = ("SAM (no attention cost)" if SAM_ATTENTION_COST == 0
       else f"SAM (attention at {SAM_ATTENTION_COST:.0%})")
SAMM = ("SAMm (matched, no attention cost)" if SAM_ATTENTION_COST == 0
        else f"SAMm (matched, attention at {SAM_ATTENTION_COST:.0%})")
FAMILIES = [(VANILLA, "vanilla"), (SAM, "sam")]        # the two families in every figure above
FAMILIES3 = FAMILIES + [(SAMM, "samm")]                # SAMm joins only in the last two

In [ ]:
def h_balanced_dims(H: int) -> dict:
    """Every dimension of an H-balanced model, derived from H alone. GQA throughout."""
    if H % 2 != 0:
        raise ValueError(f"H must be divisible by 2, got {H}")
    dmodel = 64 * H
    return dict(H=H, n_blocks=H, dmodel=dmodel, datt=dmodel, dff=4 * dmodel,
                q_heads=H, kv_heads=H // 2, head_dim=HEAD_DIM)


def weight_params(vocab_size: int, H: int, *, variant: str = "vanilla",
                  gated_ff: bool = True) -> dict:
    """Parameters that take part in a matmul, i.e. everything except the embedding table.

    variant="sam"  drops q_proj and k_proj; v_proj, o_proj and everything else are untouched.
    variant="samm" drops them too, then widens dff by exactly what they cost, so the parameter
                   count matches the vanilla model of the same H (dff = 288H with a gated FF).
    """
    if variant not in VARIANTS:
        raise ValueError(f"variant must be one of {VARIANTS}, got {variant!r}")
    d = h_balanced_dims(H)
    dmodel, datt, dff, L = d["dmodel"], d["datt"], d["dff"], d["n_blocks"]
    kv_width = d["head_dim"] * d["kv_heads"]
    q, k, v, o = dmodel * datt, dmodel * kv_width, dmodel * kv_width, datt * dmodel
    attn_per_layer = (q + k + v + o) if variant == "vanilla" else (v + o)
    ff_matrices = 3 if gated_ff else 2                     # gate + up + down
    if variant == "samm":           # spend the freed q+k on dff: 1.5d^2 / (3d) = d/2 = 32H
        dff += (q + k) // (ff_matrices * dmodel)
    ff_per_layer = ff_matrices * dmodel * dff
    return dict(
        attn_projections=L * attn_per_layer,
        feedforward=L * ff_per_layer,
        lm_head=vocab_size * dmodel,
        embedding_table=vocab_size * dmodel,  # parameters, but no FLOPs (gather)
    )


def total_params(vocab_size: int, H: int, *, variant: str = "vanilla", gated_ff: bool = True) -> int:
    """All parameters, counted as nano trains them: untied embedding and head, norms, no biases."""
    p = weight_params(vocab_size, H, variant=variant, gated_ff=gated_ff)
    d = h_balanced_dims(H)
    norms = d["n_blocks"] * 2 * d["dmodel"] + d["dmodel"]
    return sum(p.values()) + norms

In [ ]:
@dataclass(frozen=True)
class Flops:
    """FLOPs of one forward pass, split by term. All counts are for the whole pass."""

    H: int
    vocab_size: int
    context_length: int
    variant: str           # "vanilla", "sam" or "samm" (no q_proj / k_proj, no score path)
    attention_cost: float  # fraction of the vanilla score path charged to SAM
    mode: str              # "prefill" (L tokens at once) or "decode" (1 token, KV cache of L)
    tokens: int            # tokens produced by this pass
    attn_projections: int  # q/k/v/o matmuls (v/o only for SAM)
    attn_scores: int       # QK^T and A@V - the weightless, quadratic part
    softmax: int
    feedforward: int
    lm_head: int

    @property
    def total(self) -> int:
        return (self.attn_projections + self.attn_scores + self.softmax
                + self.feedforward + self.lm_head)

    @property
    def per_token(self) -> float:
        return self.total / self.tokens

    @property
    def attention_share(self) -> float:
        """Fraction of the pass spent in the quadratic score path."""
        return self.attn_scores / self.total

    def as_row(self) -> dict:
        row = dict(asdict(self))
        row.update(total=self.total, per_token=self.per_token,
                   attention_share=self.attention_share)
        return row


def inference_flops(
    vocab_size: int,
    H: int,
    context_length: int,
    *,
    mode: str = "prefill",
    variant: str = "vanilla",
    attention_cost: float = 0.0,
    gated_ff: bool = True,
    softmax_flops_per_score: int = 0,
) -> Flops:
    """FLOPs of one inference forward pass of an H-balanced model (or of SAM).

    Weights cost 2 FLOPs per active parameter per token; the attention score path is counted
    exactly over the causal (i >= j) pairs. See the markdown cell above for what is excluded.

    mode="prefill": the whole context of `context_length` tokens in one pass.
    mode="decode" : one new token attending to `context_length` cached keys (incl. itself).

    attention_cost applies to the SAM variants only: 0.0 (the default) charges no score path,
    1.0 charges the same pair work as the vanilla model, and anything between scales it.
    """
    if not 0.0 <= attention_cost <= 1.0:
        raise ValueError(f"attention_cost must be in [0, 1], got {attention_cost}")
    if context_length < 1:
        raise ValueError("context_length must be >= 1")
    d = h_balanced_dims(H)
    p = weight_params(vocab_size, H, variant=variant, gated_ff=gated_ff)

    if mode == "prefill":
        tokens = context_length
        pairs = context_length * (context_length + 1) // 2
    elif mode == "decode":
        tokens = 1
        pairs = context_length
    else:
        raise ValueError(f"mode must be 'prefill' or 'decode', got {mode!r}")

    scores_per_layer = 4 * d["datt"] * pairs  # QK^T + A@V, 2*head_dim per (query, key, head)
    cost = 1.0 if variant == "vanilla" else attention_cost
    scores = int(cost * d["n_blocks"] * scores_per_layer)

    return Flops(
        H=H,
        vocab_size=vocab_size,
        context_length=context_length,
        variant=variant,
        attention_cost=cost,
        mode=mode,
        tokens=tokens,
        attn_projections=2 * p["attn_projections"] * tokens,
        attn_scores=scores,
        softmax=int(cost * d["n_blocks"] * d["q_heads"] * pairs * softmax_flops_per_score),
        feedforward=2 * p["feedforward"] * tokens,
        lm_head=2 * p["lm_head"] * tokens,
    )

## Sanity checks

Properties the formulas must have, asserted rather than eyeballed.

In [ ]:
# 1. Weight terms are exactly 2 FLOPs per active parameter per token.
f = inference_flops(50_257, 16, 1024)
p = weight_params(50_257, 16)
assert f.attn_projections == 2 * p["attn_projections"] * 1024
assert f.feedforward == 2 * p["feedforward"] * 1024
assert f.lm_head == 2 * p["lm_head"] * 1024

# 2. Attention scores are quadratic, weight terms linear: doubling L ~4x vs exactly 2x.
a, b = inference_flops(50_257, 16, 4096), inference_flops(50_257, 16, 8192)
assert b.feedforward == 2 * a.feedforward
assert 3.9 < b.attn_scores / a.attn_scores < 4.0  # L(L+1) ratio, just under 4

# 3. A prefill of L tokens costs the same as L decode steps over the same growing cache.
L = 128
step_sum = sum(inference_flops(50_257, 8, i, mode="decode").total for i in range(1, L + 1))
assert step_sum == inference_flops(50_257, 8, L).total

# 4. SAM drops exactly q_proj and k_proj: half the attention projection work, FF and head
#    untouched, and the closed forms 55296*H^3 (SAM) / 61440*H^3 (vanilla) for non-emb params.
van = inference_flops(50_257, 24, 4096)
sam = inference_flops(50_257, 24, 4096, variant="sam")
assert sam.attn_projections * 2 == van.attn_projections
assert sam.feedforward == van.feedforward and sam.lm_head == van.lm_head
for H in (12, 16, 50):
    emb = 2 * 50_257 * 64 * H
    assert abs(total_params(50_257, H, variant="sam") - emb - 55296 * H ** 3) < 0.001 * 55296 * H ** 3
    assert abs(total_params(50_257, H) - emb - 61440 * H ** 3) < 0.001 * 61440 * H ** 3

# 4b. SAMm matches the vanilla parameter count exactly, so it does exactly the same weight
#     work per token - its only saving is the score path. Gated: dff = 288H = 4.5 dmodel.
for H in (12, 16, 50):
    assert total_params(50_257, H, variant="samm") == total_params(50_257, H)
    assert weight_params(50_257, H, variant="samm")["feedforward"] == H * 3 * 64 * H * 288 * H
    assert total_params(50_257, H, variant="samm", gated_ff=False) == \
        total_params(50_257, H, gated_ff=False)
samm = inference_flops(50_257, 24, 4096, variant="samm")
assert samm.attn_projections == sam.attn_projections and samm.attn_scores == 0
assert samm.attn_projections + samm.feedforward == van.attn_projections + van.feedforward

# 5. At attention_cost=0 SAM has no score path at all, so its cost is exactly linear in tokens;
#    at attention_cost=1 it pays the vanilla pair work.
free = [inference_flops(50_257, 16, L, variant="sam") for L in (8192, 16_384)]
assert all(f.attn_scores == 0 for f in free)
assert free[1].total == 2 * free[0].total
full = inference_flops(50_257, 16, 8192, variant="sam", attention_cost=1.0)
assert full.attn_scores == inference_flops(50_257, 16, 8192).attn_scores
assert inference_flops(50_257, 16, 8192, variant="sam", attention_cost=0.25).attn_scores == \
    full.attn_scores // 4

print("all checks pass")
for v in VARIANTS:
    f = inference_flops(VOCAB, 16, 2048, variant=v, attention_cost=SAM_ATTENTION_COST,
                        gated_ff=GATED_FF)
    print(f"H=16 {v:>7}: {total_params(VOCAB, 16, variant=v, gated_ff=GATED_FF) / 1e6:7.1f}M params | "
          f"2048-token prefill {f.total / 1e12:.2f} TFLOPs "
          f"({f.per_token / 1e9:.2f} GFLOPs/token), attention score path {f.attention_share:.1%}")

## Figures

Both model families are drawn together over the same `H` values: **colour is `H`, line style is the
family** (solid = vanilla, dashed = SAM). Every line figure carries a **log / linear switch for
its value axis** - the buttons above the plot relayout it in place, direct labels included (the
stacked bar chart has no such switch: its value axis is a share and its other axis is categorical).
The context axis carries
`CONTEXT_POINTS` log-spaced samples (100 by default) and is labelled at the powers of two, so
hovering anywhere between the labels reads out real values. One chart
surface, light or dark: set `THEME` and re-run. The series palette is the validated categorical
order, assigned to model sizes in fixed order and never cycled; the vanilla line of each size is
also labelled directly at its right end, so identity never rests on colour alone. The table under
the first figure is the same data in readable form.

In [ ]:
PALETTE = {
    "light": dict(
        surface="#fcfcfb", text="#0b0b0b", muted="#52514e", grid="#e6e5e1",
        series=["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"],
        series_long=["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300",
                     "#9a5ce0", "#00a0b0", "#b02a37", "#7a5c2e"],
    ),
    "dark": dict(
        surface="#1a1a19", text="#f0efec", muted="#c3c2b7", grid="#34332f",
        series=["#3987e5", "#d95926", "#199e70", "#c98500", "#d55181", "#008300"],
        series_long=["#3987e5", "#d95926", "#199e70", "#c98500", "#d55181", "#008300",
                     "#a970ea", "#19aab8", "#e06c75", "#b08a52"],
    ),
}
T = PALETTE[THEME]


def size_label(H, variant="vanilla"):
    """A model's parameter count in billions, two decimals, rounded up: the series name."""
    billions = total_params(VOCAB, H, variant=variant, gated_ff=GATED_FF) / 1e9
    return f"{ceil(billions * 100) / 100:.2f}B"


SIZES = {H: size_label(H) for H in H_VALUES}   # vanilla parameter count per H, used as the key


def style(fig, *, title, subtitle=None, xtitle=None, ytitle=None, legend_title=None):
    """Recessive axes, readable ink, one consistent surface."""
    head = title if subtitle is None else f"{title}<br><span style='font-size:13px;color:{T['muted']}'>{subtitle}</span>"
    fig.update_layout(
        title=dict(text=head, font=dict(size=18, color=T["text"]), x=0, xanchor="left", y=0.96,
                   yanchor="top"),
        paper_bgcolor=T["surface"], plot_bgcolor=T["surface"],
        font=dict(family="Inter, system-ui, sans-serif", size=13, color=T["muted"]),
        legend=dict(title=legend_title or "", bgcolor="rgba(0,0,0,0)",
                    font=dict(color=T["text"]), yanchor="top", y=1, xanchor="left", x=1.02),
        margin=dict(l=70, r=140, t=130, b=60), hovermode="x unified", height=520,
    )
    fig.update_xaxes(title=xtitle, showgrid=False, zeroline=False,
                     linecolor=T["grid"], ticks="outside", tickcolor=T["grid"])
    fig.update_yaxes(title=ytitle, gridcolor=T["grid"], zeroline=False, linecolor="rgba(0,0,0,0)")
    return fig


def family_legend(fig, families):
    """Colour carries the model size, dash the family: one entry per size, plus one key per family.

    `families` is [(label, dash), ...]; the first one carries the per-size legend entries.
    """
    primary = families[0][0]
    for tr in fig.data:
        h_label, family = (part.strip() for part in tr.name.split(",", 1))
        tr.name, tr.legendgroup = h_label, h_label
        tr.showlegend = family == primary
    for label, dash in families:
        fig.add_trace(go.Scatter(x=[None], y=[None], mode="lines", name=label,
                                 legendgroup=label, showlegend=True,
                                 line=dict(color=T["muted"], width=2, dash=dash)))
    return fig


def two_family_legend(fig, solid_label, dashed_label):
    """Colour carries the model size, dash the family: one legend entry per size, plus two keys."""
    return family_legend(fig, [(solid_label, "solid"), (dashed_label, "dash")])


def _stagger(ends, lo, hi, gap_frac=0.05):
    """Push labels apart so they never collide, without pushing any of them off the axis.

    Labels are placed from the top down with a minimum gap of ~5% of the plotted range; the gap
    shrinks if the cascade would otherwise run past `lo`.
    """
    span = (hi - lo) or 1.0
    gap = min(gap_frac * span, span / max(len(ends) - 1, 1))
    placed, out = None, {}
    for y_val, key in sorted(ends, reverse=True):
        y_text = y_val if placed is None else max(min(y_val, placed - gap), lo)
        out[key] = y_text
        placed = y_text
    return out


def label_line_ends(fig, df, x, y, color, *, logx=False, log_default=True,
                    fmt=lambda key: f"H={key}", colours=None):
    """Direct labels at the right end of each line, laid out for BOTH y scales.

    Plotly reads annotation coordinates as log10 of the value on a log axis, so each label gets a
    log position and a linear one; `y_scale_buttons` swaps between them.
    """
    ends, palette, xs = [], dict(colours or {}), {}
    for i, (key, part) in enumerate(df.groupby(color, sort=False)):
        last = part.sort_values(x).iloc[-1]
        ends.append((last[y], key))
        palette.setdefault(key, T["series"][i % len(T["series"])])
        xs[key] = log10(last[x]) if logx else last[x]

    lin = _stagger(ends, df[y].min(), df[y].max())
    positive = df[y][df[y] > 0]                       # a log axis cannot show a zero series
    floor = positive.min() if len(positive) else 1.0
    log = _stagger([(log10(max(v, floor)), k) for v, k in ends],
                   log10(floor), log10(max(df[y].max(), floor)))
    order = [k for _, k in sorted(ends, reverse=True)]
    for key in order:
        fig.add_annotation(x=xs[key], y=(log if log_default else lin)[key],
                           text=fmt(key), showarrow=False, xanchor="left", xshift=8,
                           font=dict(size=12, color=palette[key]))
    return {"log": [log[k] for k in order], "linear": [lin[k] for k in order]}


def y_scale_buttons(fig, positions, *, active="log", ranges=None):
    """A log / linear switch for the y axis that also moves the direct labels.

    `ranges` gives an explicit y range per scale, for axes (like a 0-100% share) whose fixed
    range would otherwise be read as decades once the axis goes log.
    """
    def args(kind):
        out = {"yaxis.type": kind, "yaxis.dtick": 1 if kind == "log" else None}
        if ranges:
            out["yaxis.range"] = ranges[kind]
        for i, y in enumerate(positions[kind]):
            out[f"annotations[{i}].y"] = y
        return [out]

    fig.update_layout(updatemenus=[dict(
        type="buttons", direction="right", showactive=True, x=0, xanchor="left",
        y=1.06, yanchor="bottom", pad=dict(t=0, b=0),
        bgcolor=T["surface"], bordercolor=T["grid"], borderwidth=1,
        font=dict(size=12, color=T["text"]), active=0 if active == "log" else 1,
        buttons=[dict(label="log y", method="relayout", args=args("log")),
                 dict(label="linear y", method="relayout", args=args("linear"))],
    )])
    fig.update_yaxes(type=active, dtick=1 if active == "log" else None,
                     **({"range": ranges[active]} if ranges else {}))
    return fig


def token_ticks(fig, values):
    """Readable ticks on a log context axis: 256, 1k, 4k, ... instead of plotly's minor ladder."""
    def tick(v):
        if v >= 1 << 20:
            return f"{v >> 20}M"
        return f"{v // 1024}k" if v >= 1024 else str(v)

    fig.update_xaxes(tickmode="array", tickvals=values, ticktext=[tick(v) for v in values],
                     minor_showgrid=False, showticklabels=True)
    return fig

In [ ]:
rows = [inference_flops(VOCAB, H, L, variant=v, attention_cost=SAM_ATTENTION_COST,
                                gated_ff=GATED_FF).as_row()
        | {"model": name}
        for name, v in FAMILIES for H in H_VALUES for L in CONTEXTS]
df = pd.DataFrame(rows)
df["TFLOPs"] = df["total"] / 1e12
df["size"] = df["H"].map(SIZES)

fig = px.line(df, x="context_length", y="TFLOPs", color="size", line_dash="model",
              log_x=True, color_discrete_sequence=T["series"],
              line_dash_map={VANILLA: "solid", SAM: "dash"},
              category_orders={"size": [SIZES[H] for H in H_VALUES], "model": [VANILLA, SAM]},
              hover_data={"size": False, "context_length": ":,", "TFLOPs": ":.3f",
                          "attention_share": ":.1%"})
fig.update_traces(line_width=2, mode="lines")
two_family_legend(fig, VANILLA, SAM)
pos = label_line_ends(fig, df[df["model"] == VANILLA], "context_length", "TFLOPs", "size",
                      logx=True, fmt=str)
token_ticks(fig, CONTEXT_TICKS)
style(fig, title="Cost of one forward pass over the full context",
      subtitle=f"GQA, gated FF, vocab {VOCAB:,} · solid = vanilla, dashed = {SAM} · log x",
      xtitle="context length (tokens)", ytitle="TFLOPs per prefill", legend_title="parameters")
y_scale_buttons(fig, pos, active="log")   # decades by default; the buttons switch to linear
fig.update_yaxes(minor_showgrid=False)
fig.show()

In [ ]:
# Table view of the same figure - accessibility fallback and the numbers to quote.
table = (df[df["context_length"].isin(CONTEXT_TICKS)]
           .pivot_table(index="context_length", columns=["model", "size"], values="TFLOPs")
           .rename_axis(index="context length").round(2))
table[[VANILLA, SAM]]

### The cost of one generated token

The figure above is a whole prefill, so it grows with the context simply because there are more
tokens in it. This one is **one decode step with a warm KV cache**: the FF, the projections and the
head run for a single token, and attention compares that token's `q` against every cached `k`,
scales all the cached `v` by those scores and sums them into one vector. The weight work is
therefore constant and the score work is `4 · datt · L` per layer - linear in the context, not
quadratic:

```
vanilla:  2 · (61440 H^3 + 64 V H)  +  256 H^2 L          FLOPs per generated token
SAM:      2 · (55296 H^3 + 64 V H)  +  c · 256 H^2 L      (c = 0 by default -> constant)
```

(the prefill's `128 H^2 L^2` carries a 1/2 from `pairs = L(L+1)/2`; a decode step attends to all
`L` cached keys, so it has no such factor)

A prefill of `L` tokens is the sum of these steps for 1..L, which is why it ends up quadratic while
each step is linear.

In [ ]:
dec = pd.DataFrame([inference_flops(VOCAB, H, L, mode="decode", variant=v,
                                    attention_cost=SAM_ATTENTION_COST,
                                    gated_ff=GATED_FF).as_row() | {"model": name}
                    for name, v in FAMILIES for H in H_VALUES for L in CONTEXTS])
dec["GFLOPs"] = dec["total"] / 1e9
dec["size"] = dec["H"].map(SIZES)

fig = px.line(dec, x="context_length", y="GFLOPs", color="size", line_dash="model",
              log_x=True, color_discrete_sequence=T["series"],
              line_dash_map={VANILLA: "solid", SAM: "dash"},
              category_orders={"size": [SIZES[H] for H in H_VALUES], "model": [VANILLA, SAM]},
              hover_data={"size": False, "context_length": ":,", "GFLOPs": ":.2f",
                          "attention_share": ":.1%"})
fig.update_traces(line_width=2, mode="lines")
two_family_legend(fig, VANILLA, SAM)
pos = label_line_ends(fig, dec[dec["model"] == VANILLA], "context_length", "GFLOPs", "size",
                      logx=True, fmt=str)
token_ticks(fig, CONTEXT_TICKS)
style(fig, title="Cost of generating one token, with the context already cached",
      subtitle=f"one decode step \u00b7 solid = vanilla, dashed = {SAM} \u00b7 log x",
      xtitle="tokens already in the KV cache", ytitle="GFLOPs per generated token",
      legend_title="parameters")
y_scale_buttons(fig, pos, active="log")
fig.show()

In [ ]:
# Same data as a table: SAM is flat in context, the vanilla model is not.
(dec[dec["context_length"].isin(CONTEXT_TICKS)]
    .pivot_table(index="context_length", columns=["model", "size"], values="GFLOPs")
    .rename_axis(index="cached tokens").round(2))[[VANILLA, SAM]]

#### The same figure as a ratio

Dividing the two lines above gives how many times more a decode step costs on the vanilla model
than on SAM of the same size. It starts at the ratio of the weights alone - `61440 / 55296 = 1.11`
once the blocks outweigh the LM head - and then grows linearly with the cache, because only the
vanilla line carries the `256 H^2 L` score term:

```
ratio(L) = (2 (61440 H^3 + 64 V H) + 256 H^2 L) / (2 (55296 H^3 + 64 V H) + c * 256 H^2 L)
         -> L / (432 H)   for a large cache at c = 0
         -> 1 / c         for a large cache at c > 0
```

So the advantage has no ceiling while `c = 0`, and a hard one of `1 / c` as soon as SAM is charged
anything for a score path - 4x at `c = 25%`, nothing left at `c = 100%`.

The smaller the model, the steeper the climb: the term SAM skips grows as `H^2` while the weights
it still pays grow as `H^3`, so at a fixed cache the ratio is larger at small sizes - the mirror
image of what the prefill sweeps show. At the default sizes every line leaves the left edge at
about **1.12** and the spread at a 1M-token cache runs from **48.6x** for the largest model to
**124.7x** for the smallest; `L / (432 H)` is already within 1% of the first of those.

In [ ]:
rat = (dec.pivot_table(index=["H", "context_length"], columns="model", values="GFLOPs")
          .reset_index())
rat["ratio"] = rat[VANILLA] / rat[SAM]
rat["size"] = rat["H"].map(SIZES)

fig = px.line(rat, x="context_length", y="ratio", color="size",
              log_x=True, color_discrete_sequence=T["series"],
              category_orders={"size": [SIZES[H] for H in H_VALUES]},
              hover_data={"size": False, "context_length": ":,", "ratio": ":.2f"})
fig.update_traces(line_width=2, mode="lines")
pos = label_line_ends(fig, rat, "context_length", "ratio", "size", logx=True, fmt=str)
token_ticks(fig, CONTEXT_TICKS)
style(fig, title="How many times a generated token costs more on vanilla than on SAM",
      subtitle=f"the two lines above, divided · {SAM} · log x",
      xtitle="tokens already in the KV cache", ytitle="vanilla GFLOPs / SAM GFLOPs",
      legend_title="parameters")
y_scale_buttons(fig, pos)
fig.show()

### Where attention starts to dominate

The weight terms are linear in tokens and the score path is quadratic, so there is a context length
`L*` at which they cost the same. Setting the two equal (GQA, gated FF, `pairs ~ L^2/2`):

```
vanilla:                   128 H^2 L^2 = L (122880 H^3 + 128 V H)  ->  L* = 960 H + V / H
SAM at attention_cost c: c 128 H^2 L^2 = L (110592 H^3 + 128 V H)  ->  L* = (864 H + V / H) / c
```

`L*` is **not** monotone in `H`. The first term is the blocks (which grow as `H^3` against the
score path's `H^2`) and the `V / H` term is the LM head (fixed vocab, so it fades as the model
grows). They trade off at `H = sqrt(V / 960)`, about **H = 7** for a 50k vocab: below that the head
dominates and attention bites early, above it every step up in `H` pushes the crossover *later*.

At the default `c = 0` **SAM has no crossover at all**: its cost is linear in tokens, so attention
never takes any share of the pass however long the context gets. That is where its advantage comes
from - the vanilla model spends most of a long pass on the quadratic term and SAM spends none of
it. Turning the knob up brings the crossover back in proportion: `c = 25%` puts it at four times
the formula's base, `c = 100%` slightly below the vanilla `L*` (the `864 H` term), because dropping
Q and K also makes the weights cheaper.

In [ ]:
share = df.copy()
share["attention_share_pct"] = share["attention_share"] * 100

fig = px.line(share, x="context_length", y="attention_share_pct", color="size",
              line_dash="model", log_x=True, color_discrete_sequence=T["series"],
              line_dash_map={VANILLA: "solid", SAM: "dash"},
              category_orders={"size": [SIZES[H] for H in H_VALUES], "model": [VANILLA, SAM]},
              hover_data={"size": False, "context_length": ":,", "attention_share_pct": ":.1f"})
fig.update_traces(line_width=2, mode="lines")
two_family_legend(fig, VANILLA, SAM)
pos = label_line_ends(fig, share[share["model"] == VANILLA], "context_length",
                      "attention_share_pct", "size", logx=True, log_default=False, fmt=str)
token_ticks(fig, CONTEXT_TICKS)
style(fig, title="Share of the forward pass spent in the attention score path",
      subtitle=f"QKᵀ and A@V only · solid = vanilla, dashed = {SAM}",
      xtitle="context length (tokens)", ytitle="% of total FLOPs", legend_title="parameters")
y_scale_buttons(fig, pos, active="linear",       # a share axis reads best linear; log is a click away
                ranges={"linear": [0, 100], "log": [-2, 2]})
fig.update_yaxes(ticksuffix="%")
fig.show()

In [ ]:
def crossover_exact(H, vocab=VOCAB, hi=1 << 22):
    lo = 1
    while lo < hi:  # attention_share is monotone in L, so bisect it
        mid = (lo + hi) // 2
        if inference_flops(vocab, H, mid, gated_ff=GATED_FF).attention_share < 0.5:
            lo = mid + 1
        else:
            hi = mid
    return lo


def sam_crossover(H, c):
    return "never" if c == 0 else f"{round((864 * H + VOCAB / H) / c):,}"


pd.DataFrame([{"H": H, "params": size_label(H),
               "vanilla L* = 960H + V/H": round(960 * H + VOCAB / H),
               "vanilla L* (exact)": crossover_exact(H),
               f"SAM L* at c={SAM_ATTENTION_COST:.0%}": sam_crossover(H, SAM_ATTENTION_COST),
               "SAM L* at c=25%": sam_crossover(H, 0.25),
               "params saved": f"{1 - total_params(VOCAB, H, variant='sam', gated_ff=GATED_FF) / total_params(VOCAB, H, gated_ff=GATED_FF):.1%}"}
              for H in [4, 8] + H_VALUES])  # 4 and 8 included to show the minimum near H = 7

### What the FLOPs are spent on

Same model size, four context lengths, each shown for both families. Everything except the score
path is linear in the context; SAM removes a slice of the attention projections and, at
`SAM_ATTENTION_COST = 0`, the score path entirely - which is why the vanilla bar is almost all
attention at 1M tokens while the SAM bar has none.

In [ ]:
TERMS = ["feedforward", "attn_projections", "lm_head", "attn_scores"]
LABELS = {"feedforward": "gated feed-forward (gate, up, down)",
          "attn_projections": "attention projections",
          "lm_head": "LM head", "attn_scores": "attention scores (QKᵀ, A@V)"}

passes = {(L, name): inference_flops(VOCAB, H_PICK, L, variant=v,
                                     attention_cost=SAM_ATTENTION_COST, gated_ff=GATED_FF)
          for L in CONTEXTS_BD for name, v in FAMILIES}
order = [f"{L:,} · {name}" for L in CONTEXTS_BD for name, _ in FAMILIES]
bd = pd.DataFrame([
    {"row": f"{L:,} · {name}", "term": LABELS[t],
     "share": getattr(f, t) / f.total * 100, "TFLOPs": getattr(f, t) / 1e12}
    for (L, name), f in passes.items() for t in TERMS
])

fig = px.bar(bd, x="share", y="row", color="term", orientation="h",
             color_discrete_sequence=T["series"],
             category_orders={"term": [LABELS[t] for t in TERMS], "row": order},
             hover_data={"row": False, "share": ":.1f", "TFLOPs": ":.2f"})
fig.update_traces(marker_line_color=T["surface"], marker_line_width=2)  # 2px surface gap
for (L, name), f in passes.items():  # absolute cost, since shares alone hide the growth
    fig.add_annotation(x=100, y=f"{L:,} · {name}", text=f"{f.total / 1e12:,.1f} TFLOPs",
                       showarrow=False, xanchor="left", xshift=10,
                       font=dict(size=12, color=T["text"]))
style(fig, title=f"Where a prefill pass spends its FLOPs ({size_label(H_PICK)} model)",
      subtitle=f"dmodel {64 * H_PICK}, {H_PICK} layers, GQA, gated FF · "
               f"vanilla vs {SAM} at {size_label(H_PICK, 'sam')}",
      xtitle="% of the pass", ytitle="context length · model", legend_title="")
fig.update_layout(barmode="stack", hovermode="closest", height=520,
                  margin=dict(l=170, r=150, t=100, b=120),
                  legend=dict(orientation="h", x=0, y=-0.22, yanchor="top"))
fig.update_xaxes(range=[0, 100], ticksuffix="%", showgrid=False)  # a share axis: linear only
fig.show()

### Turning the attention knob

One size (`H = 50`), and SAM charged for 1, 2, 4, 8, 16 and 32 percent of the vanilla score path,
with the vanilla model itself for reference.

Each doubling of `c` doubles the quadratic term, but what that does to the *total* depends on which
term is winning. Up to about 32k tokens all six sit on the weights-only floor and on top of each
other - at 32k even `c = 32%` is only 1.24x the floor. Past that they fan out, and at 1M the
quadratic term owns the pass: 1.2x the floor at `c = 1%`, 8.6x at 32%, against the vanilla model's
25x. In share-of-vanilla terms that is 5.0% of a vanilla prefill at `c = 1%` and 34.6% at 32%.

In [ ]:
REFERENCE = "vanilla"


def cost_sweep(h_sam, h_ref):
    """SAM at a range of attention costs against a vanilla model, both at a chosen size."""
    rows = ([inference_flops(VOCAB, h_sam, L, variant="sam", attention_cost=c,
                             gated_ff=GATED_FF).as_row() | {"cost": f"{c:.0%}"}
             for c in SWEEP_COSTS for L in CONTEXTS]
            + [inference_flops(VOCAB, h_ref, L, gated_ff=GATED_FF).as_row() | {"cost": REFERENCE}
               for L in CONTEXTS])
    data = pd.DataFrame(rows)
    data["TFLOPs"] = data["total"] / 1e12

    order = [f"{c:.0%}" for c in SWEEP_COSTS] + [REFERENCE]
    colours = {label: T["series"][i] for i, label in enumerate(order[:-1])} | {REFERENCE: T["muted"]}
    ref_label = f"vanilla {size_label(h_ref)}"

    fig = px.line(data, x="context_length", y="TFLOPs", color="cost", log_x=True,
                  color_discrete_map=colours, category_orders={"cost": order},
                  hover_data={"cost": False, "context_length": ":,", "TFLOPs": ":.2f",
                              "attention_share": ":.1%"})
    fig.update_traces(line_width=2, mode="lines")
    for tr in fig.data:  # the reference stays solid, the SAM lines are dashed as everywhere else
        tr.line.dash = "solid" if tr.name == REFERENCE else "dash"
        if tr.name == REFERENCE:
            tr.name = ref_label
    pos = label_line_ends(fig, data, "context_length", "TFLOPs", "cost", logx=True,
                          colours=colours,
                          fmt=lambda key: ref_label if key == REFERENCE else f"c={key}")
    token_ticks(fig, CONTEXT_TICKS)
    same = h_sam == h_ref
    style(fig,
          title=(f"What the attention-cost knob buys ({size_label(h_sam, 'sam')} SAM)" if same
                 else f"SAM {size_label(h_sam, 'sam')} against vanilla {size_label(h_ref)}"),
          subtitle=(f"SAM charged {SWEEP_COSTS[0]:.0%}-{SWEEP_COSTS[-1]:.0%} of the vanilla score "
                    f"path \u00b7 dashed = SAM, solid = vanilla \u00b7 log x"),
          xtitle="context length (tokens)", ytitle="TFLOPs per prefill",
          legend_title="attention cost")
    return y_scale_buttons(fig, pos, active="log")


cost_sweep(*SWEEP_PAIRS[0]).show()

### A bigger SAM against a smaller vanilla model

SAM at `H = 50` carries 7.23B parameters against vanilla `H = 40`'s 4.19B - **1.73x the model** -
so it starts out more expensive: 57.9 against 36.7 TFLOPs for a 4k prefill. The quadratic term
overturns that at around **29k tokens** (34k at `c = 8%`, 59k at `c = 32%`), and by 1M the bigger
SAM model costs 14,833 TFLOPs against the smaller vanilla model's 233,696 - 16x less for 73% more
parameters.

In [ ]:
cost_sweep(*SWEEP_PAIRS[1]).show()

### The same comparison one size class up

`H = 100` SAM (55.94B parameters) against `H = 80` vanilla (31.97B), again 1.75x the model. The
pattern repeats with everything shifted right, because both the weight gap and the score path grow
with `H`: the crossover moves to about **58k tokens** (67k at `c = 8%`, 117k at `c = 32%`), and at
1M it is 116,639 against 967,231 TFLOPs, a factor of 8.3.

In [ ]:
cost_sweep(*SWEEP_PAIRS[2]).show()

### And the same again at the 1T scale

`H = 252` SAM (886.53B parameters) against `H = 200` vanilla (492.81B) - 1.8x the model, the widest
parameter gap of the three pairs, and a 4k prefill costs 7,256 against 4,118 TFLOPs. The crossover
moves out to about **154k tokens** (176k at `c = 8%`, 312k at `c = 32%`), and at 1M it is 1,857,475
against 6,661,646 TFLOPs, a factor of 3.6.

That factor shrinks with every size class - 16x, 8.3x, 3.6x - even though the contexts and the cost
settings are identical. Weight work grows as `H³` and the score path only as `H²`, so the term
SAM skips is a smaller slice of a bigger model, and the extra parameters SAM carries are a larger
one. Scale alone erodes the advantage; only longer contexts restore it.

In [ ]:
cost_sweep(*SWEEP_PAIRS[3]).show()

### SAMm: the same parameters, none of the score path

`SAMm` is SAM with the freed `q_proj`/`k_proj` weights put back into the feed-forward rather than
returned - `dff = 288 H` instead of `256 H`, i.e. `4.5 dmodel` - which matches the vanilla
parameter count **exactly, to the parameter**, at every `H`
(see [`../concepts/sam-model.md`](../concepts/sam-model.md#samm--the-parameter-matched-variant)).

So all three families are drawn here against the same colour, and the colour is the *vanilla*
parameter count: SAMm has precisely that count, SAM about 10% less. Same parameters means the same
weight work per token, so SAMm and the vanilla model sit on top of each other until the score path
starts to matter, and from there the gap between them **is** the score path.

In [ ]:
tri = pd.DataFrame([inference_flops(VOCAB, H, L, variant=v, attention_cost=SAM_ATTENTION_COST,
                                    gated_ff=GATED_FF).as_row() | {"model": name}
                    for name, v in FAMILIES3 for H in H_VALUES for L in CONTEXTS])
tri["TFLOPs"] = tri["total"] / 1e12
tri["size"] = tri["H"].map(SIZES)

DASHES = {VANILLA: "solid", SAM: "dash", SAMM: "dot"}
fig = px.line(tri, x="context_length", y="TFLOPs", color="size", line_dash="model",
              log_x=True, color_discrete_sequence=T["series"], line_dash_map=DASHES,
              category_orders={"size": [SIZES[H] for H in H_VALUES],
                               "model": [VANILLA, SAM, SAMM]},
              hover_data={"size": False, "context_length": ":,", "TFLOPs": ":.3f",
                          "attention_share": ":.1%"})
fig.update_traces(line_width=2, mode="lines")
family_legend(fig, list(DASHES.items()))
pos = label_line_ends(fig, tri[tri["model"] == VANILLA], "context_length", "TFLOPs", "size",
                      logx=True, fmt=str)
token_ticks(fig, CONTEXT_TICKS)
style(fig, title="All three families over the same context",
      subtitle=f"colour = the vanilla parameter count \u00b7 solid = vanilla, dashed = {SAM}, "
               f"dotted = {SAMM} \u00b7 log x",
      xtitle="context length (tokens)", ytitle="TFLOPs per prefill", legend_title="parameters")
y_scale_buttons(fig, pos, active="log")
fig.update_yaxes(minor_showgrid=False)
fig.show()

#### The same thing as a ratio

Each SAM line divided by the vanilla line of the same `H` and the same context. Two readings the
absolute figure cannot give:

* **The SAMm curve is the vanilla model's weight share**, `1 - attention_share`, because the two
  models do identical weight work. It therefore passes through exactly **50% at the crossover**
  `L* = 960H + V/H` - 18,501 tokens at `H = 16`, 49,005 at `H = 50` - which is the same number the
  crossover table above prints, read off a curve.
* **The vertical gap between the dotted and dashed line of one colour is SAM's parameter saving**,
  and it is constant in the context: 91.7% of the SAMm cost at `H = 16`, 90.2% at `H = 50`,
  tending to 90% as `H` grows. Everything else the two share.

At 1M tokens and the default sizes both curves are deep in single digits - SAMm 1.7% of a vanilla
prefill at `H = 16` and 4.5% at `H = 50`, SAM 1.6% and 4.0% - and the ratio *rises* with model
size, because the weight work SAMm keeps grows as `H^3` while the score path it skips grows only
as `H^2`.

In [ ]:
ratio = tri.pivot_table(index=["H", "size", "context_length"], columns="model",
                        values="TFLOPs").reset_index()
ratio = ratio.melt(id_vars=["H", "size", "context_length", VANILLA], var_name="model",
                   value_name="TFLOPs")
ratio["% of vanilla"] = ratio["TFLOPs"] / ratio[VANILLA] * 100

fig = px.line(ratio, x="context_length", y="% of vanilla", color="size", line_dash="model",
              log_x=True, color_discrete_sequence=T["series"],
              line_dash_map={SAM: "dash", SAMM: "dot"},
              category_orders={"size": [SIZES[H] for H in H_VALUES], "model": [SAMM, SAM]},
              hover_data={"size": False, "context_length": ":,", "% of vanilla": ":.2f",
                          "TFLOPs": ":.2f"})
fig.update_traces(line_width=2, mode="lines")
family_legend(fig, [(SAMM, "dot"), (SAM, "dash")])
pos = label_line_ends(fig, ratio[ratio["model"] == SAMM], "context_length", "% of vanilla", "size",
                      logx=True, log_default=False, fmt=str)
token_ticks(fig, CONTEXT_TICKS)
if ratio["% of vanilla"].min() < 50 < ratio["% of vanilla"].max():   # only if a curve crosses it
    fig.add_hline(y=50, line_dash="dot", line_color=T["muted"], line_width=1,
                  annotation_text="half the vanilla cost", annotation_position="bottom left",
                  annotation_font=dict(size=11, color=T["muted"]))
_weight_share = (" \u00b7 the dotted line is the vanilla weight share, 50% at L* = 960H + V/H"
                 if SAM_ATTENTION_COST == 0 else "")   # true only while SAMm pays no score path
style(fig, title="Cost of SAM and SAMm as a share of the vanilla model of the same H",
      subtitle=f"dotted = {SAMM}, dashed = {SAM}{_weight_share} \u00b7 log x",
      xtitle="context length (tokens)", ytitle="% of the vanilla prefill", legend_title="parameters")
y_scale_buttons(fig, pos, active="linear")
fig.update_yaxes(ticksuffix="%", minor_showgrid=False)
fig.show()

### Against a vanilla model that does not run all-to-all attention

Every figure so far charges the vanilla model for **every** causal pair, which no deployed
long-context model actually does. [`../notes/long-context-attention.md`](../notes/long-context-attention.md)
works through what the published alternatives cost; each one reduces to a **retention** `c(L)` -
the fraction of the `L(L+1)/2` pairs it still scores - and that is the same quantity as this
notebook's `SAM_ATTENTION_COST`, so it drops straight into the model above.

The ten solid curves below are the vanilla model under each of those methods, at their published
hyperparameters. The two references are **SAM at `H = 100`** (dashed) and **SAMm at `H = 80`**
(dotted), the latter parameter-matched to the vanilla model being drawn, so it is that model's
weights-only floor.

**What each line is**, in plotting order, with the retention it ends at:

| line | what it does | `c` at 1M |
|---|---|---:|
| dense | every causal pair, `L(L+1)/2` of them. FlashAttention and Ring Attention compute exactly this - they change memory traffic and parallel decomposition, not FLOPs | 100% |
| Gemma 2 1:1 | alternating layers: one sliding window of 4096, one full-context. The window term dies as `2w/L`, the global layers do not, so it settles at half | 50.39% |
| linear 3:1 | three layers of linear attention (zero pairs, a fixed-size recurrent state) per one full-attention layer | 25.00% |
| MoBA | blocks of 4096, each query routed to its top 12 by a parameter-free gate, 29 of 32 layers sparse. The 3 full layers contribute 9.4 of the 17.7 points | 17.67% |
| Gemma 3 5:1 | five local layers of window 1024 per global layer - the window shrank 4x and the ratio went 1:1 -> 5:1, both reported at minimal perplexity cost | 16.83% |
| linear 7:1 | seven linear layers per softmax layer. Jamba is literally this (1 attention per 8); MiniMax-01 phrases it as one softmax block after every seven | 12.50% |
| NSA | three gated branches: stride-16 compressed blocks (a quadratic term divided by 16, hence the floor), top-16 selected blocks of 64, and a window of 512. Trained in from scratch | 6.54% |
| DSA | a small FP8 "lightning indexer" scores every pair cheaply (5.9% of the dense cost, and that is the floor), then full attention runs over the top 2048 only | 6.29% |
| MInference | one of three patterns per head, fixed offline: A-shape, vertical-slash, or block-sparse - about 4k columns and diagonals per query. A serving-time retrofit, no retraining | 0.78% |
| local only | a 1024 window in every layer and no global layer: linear in `L`, and the only line with nothing left to pay asymptotically | 0.20% |

The legend names models that ship each one, from their model cards and technical reports as of
2026-10-20. Two caveats belong with those names: MInference is a retrofit onto an already-trained
model rather than a model family, and MiniMax-M2 is listed under *dense* because MiniMax is
reported to have dropped linear attention after M1.

#### Where Mamba and the other SSMs sit

They are not a separate line because they would overplot the dotted one. A Mamba/SSM layer scores
**no pairs at all** - `c = 0`, the same slot SAM occupies - so a pure SSM's curve *is* the
weights-only floor. What it pays instead is a per-token constant: the recurrence on a fixed state
of `d_inner x N` costs about `5 d_inner N ~ 10 d N` FLOPs per token per layer, which against an
H-balanced model's `122880 H^3` of weight work per token is exactly `N / (192 H)`:

| state `N` | share of the weight work at `H = 80` |
|---|---:|
| 16 (Mamba-1) | 0.10% |
| 128 (Mamba-2) | 0.83% |
| 256 | 1.67% |

Under 2% of the pass and flat in the context, so in FLOP terms sequence mixing is nearly free: a
prefill is linear in `L` (the SSD form does matmuls inside chunks of <=128 and a recurrence between
them) and a decode step is **constant** in `L` rather than linear. The saving our charts cannot
show is the memory: the state is `256 N H^2` bytes, i.e. **210 MB per sequence at `H = 80`,
`N = 128`, and it does not grow**, against a KV cache of 800 KiB per token - 859 GB at 1M tokens.

What a fixed state costs is retrieval capacity, not FLOPs, which is why nothing ships pure: Jamba
is 1 attention layer per 8 (the `linear 7:1` line exactly), Nemotron-H keeps ~8% of its layers as
attention (4 of 52 at 8B, 10 of 118 at 56B, so just below that line), and Qwen3-Next and Kimi K3
use 3:1 with gated-delta-rule linear attention rather than an SSM. Note what this means for SAM:
**a pure SSM and SAM have the same cost curve** - strictly linear in tokens, no quadratic term -
so they also share the same open question, whether a fixed-size state can do long-range retrieval.

Quality is **not** modelled in any of this: nine of the ten lines are approximations of attention
(only the dense line is exact), while SAM's `c = 0` is an architectural claim. This figure answers
the narrower question of what each one costs.

In [ ]:
def _pairs(L):
    return L * (L + 1) / 2


def _window(L, w):      # a sliding window of w keys per query
    return _pairs(L) if L <= w else L * w - w * (w - 1) / 2


def _budget(L, b):      # a fixed budget of b keys per query (top-k selection)
    return _pairs(L) if L <= b else b * (b + 1) / 2 + (L - b) * b


# (short label, models that ship it, what it does, retention c(L)). The model attributions are
# from model cards and technical reports as of 2026-10-20 and are sourced one by one in
# ../notes/long-context-attention.md, together with the papers and the quality evidence.
ATTENTION_METHODS = [
    ("dense", "Llama 3.1, DeepSeek-V3.1, MiniMax-M2",
     "every causal pair - FlashAttention and Ring Attention compute this exactly",
     lambda L: 1.0),
    ("Gemma 2 1:1", "Gemma 2",
     "1 local : 1 global layer, window 4096",
     lambda L: (_window(L, 4096) / _pairs(L) + 1) / 2),
    ("Gemma 3 5:1", "Gemma 3, Gemma 4",
     "5 local : 1 global layer, window 1024",
     lambda L: (5 * _window(L, 1024) / _pairs(L) + 1) / 6),
    ("linear 7:1", "MiniMax-01, MiniMax-M1, Jamba",
     "7 linear-attention layers : 1 softmax layer",
     lambda L: 1 / 8),
    ("linear 3:1", "Kimi Linear, Kimi K3, Qwen3-Next",
     "3 linear-attention layers : 1 full-attention layer",
     lambda L: 1 / 4),
    ("MoBA", "Kimi",
     "blocks of 4096, top-12 per query, 29 of 32 layers sparse",
     lambda L: (29 * _budget(L, 12 * 4096) / _pairs(L) + 3) / 32),
    ("NSA", "DeepSeek 27B research model",
     "stride-16 compression + 16x64 selection + window 512",
     lambda L: min(1.0, 1 / 16 + _budget(L, 16 * 64 + 512) / _pairs(L))),
    ("DSA", "DeepSeek-V3.2, GLM-5, GLM-5.2, GLM-5.3",
     "top-2048 selection + a lightning indexer at 5.9% of the dense pair cost",
     lambda L: min(1.0, 0.059 + _budget(L, 2048) / _pairs(L))),
    ("MInference", "retrofit onto Yi-200K, GLM-4-9B-1M",
     "~4k columns and diagonals per query, chosen offline per head",
     lambda L: _budget(L, 4096) / _pairs(L)),
    ("local only", "Mistral 7B, Character.AI",
     "sliding window 1024, no global layer (those two use wider windows)",
     lambda L: _window(L, 1024) / _pairs(L)),
]


def sparse_prefill(H, L, c):
    """A vanilla prefill that scores only the fraction `c` of the causal pairs."""
    f = inference_flops(VOCAB, H, L, gated_ff=GATED_FF)
    return f.total - (1 - c) * f.attn_scores


def sam_prefill(H, L, variant="sam"):
    return inference_flops(VOCAB, H, L, variant=variant, attention_cost=SAM_ATTENTION_COST,
                           gated_ff=GATED_FF).total

In [ ]:
H_SAM, H_REF = METHOD_PAIR
SAM_LINE = f"SAM {size_label(H_SAM, 'sam')}"
SAMM_LINE = f"SAMm {size_label(H_REF, 'samm')}"   # parameter-matched to the vanilla reference

# two references in neutral ink: the bigger SAM dashed, the matched SAMm dotted
SAM_LINES = {SAM_LINE: (H_SAM, "sam", "dash", T["muted"]),
             SAMM_LINE: (H_REF, "samm", "dot", T["text"])}

# the legend names the models that ship each method; the line ends keep the short label
LEGEND = {short: f"{short} ({models})" for short, models, _, _ in ATTENTION_METHODS}
SHORT = {name: short for short, name in LEGEND.items()} | {k: k for k in SAM_LINES}

methods = pd.DataFrame([{"method": LEGEND[short], "detail": detail, "context_length": L,
                         "retention": c(L), "TFLOPs": sparse_prefill(H_REF, L, c(L)) / 1e12}
                        for short, _, detail, c in ATTENTION_METHODS for L in CONTEXTS]
                       + [{"method": name, "detail": f"{name}, {SAM}", "context_length": L,
                           "retention": float("nan"), "TFLOPs": sam_prefill(H, L, v) / 1e12}
                          for name, (H, v, _, _) in SAM_LINES.items() for L in CONTEXTS])

order = [LEGEND[short] for short, _, _, _ in ATTENTION_METHODS] + list(SAM_LINES)
colours = ({LEGEND[short]: T["series_long"][i]
            for i, (short, _, _, _) in enumerate(ATTENTION_METHODS)}
           | {name: colour for name, (_, _, _, colour) in SAM_LINES.items()})

fig = px.line(methods, x="context_length", y="TFLOPs", color="method", log_x=True,
              color_discrete_map=colours, category_orders={"method": order},
              hover_data={"method": False, "detail": True, "context_length": ":,",
                          "TFLOPs": ":.1f", "retention": ":.2%"})
fig.update_traces(line_width=2, mode="lines")
for tr in fig.data:      # the SAM lines are the references, as in every other figure here
    tr.line.dash = SAM_LINES[tr.name][2] if tr.name in SAM_LINES else "solid"
pos = label_line_ends(fig, methods, "context_length", "TFLOPs", "method", logx=True,
                      colours=colours, fmt=SHORT.get)
token_ticks(fig, CONTEXT_TICKS)
style(fig, title=f"Long-context attention methods on vanilla {size_label(H_REF)}, "
                 f"against {' and '.join(SAM_LINES)}",
      subtitle="each solid line is the vanilla model scoring only the pairs its method keeps "
               "\u00b7 dashed = SAM, dotted = the parameter-matched SAMm \u00b7 log x",
      xtitle="context length (tokens)", ytitle="TFLOPs per prefill",
      legend_title="attention (models that ship it)")
fig.update_layout(margin=dict(l=70, r=360, t=130, b=60))   # room for the labels and the legend
y_scale_buttons(fig, pos, active="log")
fig.update_yaxes(minor_showgrid=False)
fig.show()

#### Where SAM still wins

SAM at `H = 100` does 1.75x the weight work of vanilla `H = 80` per token, and that gap is
constant in the context, so SAM only comes out ahead once the vanilla model's score path is worth
at least 0.75x its own weight bill. For a method with a fixed per-query budget `B` that ratio is
`B / B*` with `B* = 480H + V/(2H)` (38,714 tokens at `H = 80`), so **SAM needs the vanilla model to
be scoring more than about 29,200 keys per query** - and no deployed budget in the literature is
anywhere near that.

The consequence is in the last column of the table: against dense attention SAM wins past 58k
tokens, against the trained-in sparse designs only past 885k (NSA) and 920k (DSA), and against
MInference-grade prefill sparsity or a plain 1k window **never** - at 1M those two cost 73,533 and
68,269 TFLOPs against SAM's 116,639. The three sweeps above are therefore the optimistic reading
of SAM's long-context case; this is the pessimistic one.

The dotted line is the other end of it, and it reads more directly than a second SAM line would.
SAMm at `H = 80` carries exactly the reference's parameters, so it does exactly the reference's
weight work and none of its score path: **the dotted line is the weights-only floor of every solid
line**, and the vertical gap up to a solid line is precisely that method's attention bill.

That floor is 66,510 TFLOPs at 1M. Dense attention sits 14.5x above it, Gemma 2's 1:1 schedule
7.8x, the 3:1 linear hybrids 4.4x, Gemma 3's 5:1 3.3x, NSA 1.89x, DSA 1.85x, MInference 1.11x and
a plain 1k window **1.03x**. Read the other way: the most aggressive published sparsity has already
brought a same-parameter vanilla model to within a few percent of paying no attention cost at all,
so at those settings there is no FLOP argument left to make for SAMm - only a quality one.

In [ ]:
BEYOND = 1 << 24    # the search cap for the crossover column: ~16.8M tokens


def sam_beats(c_fn, h_sam=None, h_ref=None):
    """Smallest context at which SAM is the cheaper of the two, by bisection."""
    h_sam, h_ref = h_sam or H_SAM, h_ref or H_REF
    if sam_prefill(h_sam, BEYOND) >= sparse_prefill(h_ref, BEYOND, c_fn(BEYOND)):
        return f"> {BEYOND / 1e6:.1f}M"
    lo, hi = 1, BEYOND
    while lo < hi:
        mid = (lo + hi) // 2
        if sam_prefill(h_sam, mid) >= sparse_prefill(h_ref, mid, c_fn(mid)):
            lo = mid + 1
        else:
            hi = mid
    return f"{lo:,}"


LONG = CONTEXT_RANGE[1]
pd.DataFrame([{"method": f"{short} - {detail}",
               "models": models,
               "c at 200k": f"{c(200_000):.2%}",
               f"c at {LONG:,}": f"{c(LONG):.2%}",
               f"vanilla {size_label(H_REF)} TFLOPs": round(sparse_prefill(H_REF, LONG, c(LONG)) / 1e12),
               f"x SAM {size_label(H_SAM, 'sam')}": round(sparse_prefill(H_REF, LONG, c(LONG))
                                                          / sam_prefill(H_SAM, LONG), 2),
               "SAM cheaper past": sam_beats(c)}
              for short, models, detail, c in ATTENTION_METHODS])

## Using it elsewhere

```python
from h_balanced_flops import inference_flops, total_params   # after `jupyter nbconvert --to script`
```

or copy the functions - they have no dependencies beyond the standard library.

* **SAM**: `variant="sam"` on `inference_flops` and `total_params`. It drops `q_proj` and
  `k_proj`, and charges no score path unless you ask for one: `attention_cost=0.25` bills a
  quarter of the vanilla pair work, `1.0` bills all of it.
* **SAMm**: `variant="samm"` - the same, with `dff` widened by `d/2` so the parameter count matches
  the vanilla model of the same `H` exactly (`288 H` with a gated FF, `304 H` without).
* **One generated token** with a warm KV cache: `inference_flops(V, H, L, mode="decode")` - the
  cost of producing the token at position `L`, not of the whole context. A prefill of `L` tokens
  equals the sum of the `L` decode steps (asserted above), so it is ~`L` times one step.
* **Ungated FF** (`up` + `down` only, as the current `def_300`/`def_800` configs are written):
  `gated_ff=False`.
* **Count the softmax**: `softmax_flops_per_score=5` (exp + max-subtract + sum + divide, roughly).

A caveat worth repeating when these numbers meet a wall-clock measurement: FLOPs are not time.
The score path is memory-bound at long contexts and the decode step is bandwidth-bound almost
everywhere, so a model with half the FLOPs does not run twice as fast.